In [1]:
import os
import sys 
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)
ROOT = Path.cwd().parent.parent.parent.resolve()

print(f"ROOT: {ROOT}")
sys.path.append(str(ROOT))

ROOT: /users/devin/Dev/financial-document-based-agent-system


In [2]:
from dochandler.main import ExTrRAGDocHandler
from cgcore.vectordb.milvus import MilvusDB
from cgcore.embedder.openai import OpenAIEmbedder
from cgcore.llm.openai import OpenAILlm

from cgcore.configs.vectordb.milvus import MilvusConfig
from cgcore.configs.embedder.openai import OpenAIEmbedderConfig
from cgcore.configs.llm.openai import OpenAILlmConfig

In [3]:
llm_config = OpenAILlmConfig(api_key=os.getenv('OPENAI_API_KEY'))
embedder_config = OpenAIEmbedderConfig(api_key=os.getenv('OPENAI_API_KEY'), model='text-davinci-003', dimesion=os.getenv('MONGO_DB_DIMENSION'))
vectordb_config = MilvusConfig(
                collection_name=os.getenv('MILVUS_COLLECTION_NAME'),
                dimensions=1536,  # Set explicit dimension value
                )


In [4]:
openai = OpenAILlm(llm_config)
embeder = OpenAIEmbedder(embedder_config)
vectordb = MilvusDB(vectordb_config)

MilvusClient connected.
pymilvus ORM connected to localhost:19530 for setup.
Collection 'financial_documents' already exists. Skipping creation.


/tmp/ipykernel_3369494/1873515904.py:1: UserWarning: Parameters {'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  openai = OpenAILlm(llm_config)
/users/devin/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/embeddings/base.py:313: UserWarning: WARNING! encoding_format is not default parameter.
                    encoding_format was transferred to model_kwargs.
                    Please confirm that encoding_format is what you intended.
  warnings.warn(


## ExTr RAG 

In [5]:
extr_rag = ExTrRAGDocHandler(
    llm=openai,
    embedder=embeder,
    db=vectordb,
    memory="none",
    history=True
)

## Document Loading test


In [6]:
extr_rag.loader.dry_run = False

In [7]:
pdf_path = "../../../data/annual-review-2024-en.pdf"

In [8]:
print(f"Testing Processing for: {os.path.basename(pdf_path)} ---")

Testing Processing for: annual-review-2024-en.pdf ---


### Questions generation and saving in the VDB

In [ ]:
records = extr_rag.loader.extr_load(pdf_path)


if len(records) > 0:
    first_record = records[0]
    
    # Structure & IDs
    print(f"Chunk ID: {first_record.get('id', 'MISSING')}")
    
    # The Chunk Content
    content_preview = first_record.get('documents', '')[:150].replace('\n', ' ')
    print(f"\n Content Preview:\n'{content_preview}...'")
    
    # The Questions
    questions = first_record.get('questions', [])
    print(f"\n Generated Questions ({len(questions)}):")
    for q in questions:
        print(f"   - {q}")
        
    # The Embeddings
    embedding = first_record.get('embeddings', [])
    emb_array = np.array(embedding)
    print(f"\n Embedding Shape: {emb_array.shape} (First 3 val: {emb_array[:3]})")

    # Milvus Compatibility
    required_keys = ['_id', 'content', 'text_embedding', 'meta_data']
    missing_keys = [key for key in required_keys if key not in first_record]
    if missing_keys:
        print(f"\n WARNING: Missing keys for Milvus Schema: {missing_keys}")
    else:
        print("\n SCHEMA CHECK: All keys ready for Milvus!")

else:
    print(" Error: No records were returned. Check Loader/Chunker.")